# 06 — silver_order_reviews
**Entrée :** `bronze.order_reviews` — **Sortie :** `silver.silver_order_reviews` (**99 224 lignes**, option « garder et marquer »)
**Pourquoi cette option :** elle respecte la règle « Silver = Bronze ». Le dédoublonnage (un avis par commande) se fait **en Gold**.
**Ajouts :** `is_duplicate_review_id`, `review_rank` (1 = avis le plus récent de la commande), `has_comment`.

## Partie A — Diagnostic R18 (les 789 review_id en double)
Les 5 nombres et les 3 cas à documenter dans le README.

In [0]:
CATALOG = "workspace"
BRONZE = f"{CATALOG}.bronze"
SILVER = f"{CATALOG}.silver"
# Les tables Bronze sont supposées s'appeler : orders, customers, order_items, order_payments,
# order_reviews, products, sellers, product_category_name_translation.
# Si les tiennes ont d'autres noms, change-les dans les requêtes (Ctrl+F).

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")


def scalar(sql):
    """Exécute une requête qui renvoie une seule valeur."""
    return spark.sql(sql).collect()[0][0]


def check(label, obtenu, attendu):
    """Compare ce qu'on obtient à ce qu'on attend (valeurs de ton document de travail)."""
    ok = obtenu == attendu
    print(("✅ OK     " if ok else "❌ ÉCART  ") + f"{label} : obtenu = {obtenu} | attendu = {attendu}")

In [0]:
b = f"{BRONZE}.order_reviews"
lignes = scalar(f"SELECT COUNT(*) FROM {b}")
ids = scalar(f"SELECT COUNT(DISTINCT review_id) FROM {b}")
cles_dup = scalar(f"SELECT COUNT(*) FROM (SELECT review_id FROM {b} GROUP BY review_id HAVING COUNT(*) > 1)")
print("1. lignes                      :", lignes)
print("2. review_id distincts         :", ids)
print("3. lignes en trop (1 - 2)      :", lignes - ids, "(attendu 814)")
print("4. review_id concernés         :", cles_dup, "(attendu 789)")
print("5. lignes dans ces doublons    :", scalar(f"""
    SELECT COUNT(*) FROM {b} WHERE review_id IN
    (SELECT review_id FROM {b} GROUP BY review_id HAVING COUNT(*) > 1)"""))

### Les 3 cas : pour chaque `review_id` en double, les lignes concernent-elles des commandes différentes ?

In [0]:
spark.sql(f"""
    WITH d AS (
        SELECT review_id, COUNT(*) AS nb_lignes, COUNT(DISTINCT order_id) AS nb_commandes,
               COUNT(DISTINCT CONCAT_WS('|', review_score, review_comment_title, review_comment_message,
                                        review_creation_date, review_answer_timestamp)) AS nb_contenus
        FROM {b} GROUP BY review_id HAVING COUNT(*) > 1)
    SELECT CASE WHEN nb_commandes > 1 AND nb_contenus = 1 THEN 'A : même avis recopié sur plusieurs commandes'
                WHEN nb_commandes > 1 THEN 'B : même identifiant, contenus différents, commandes différentes'
                ELSE 'C : même commande, lignes répétées' END AS cas,
           COUNT(*) AS nb_review_id, SUM(nb_lignes) AS nb_lignes
    FROM d GROUP BY 1 ORDER BY 1
""").show(truncate=False)

## Partie B — Construire silver_order_reviews

In [0]:
spark.sql(f"""
CREATE OR REPLACE TABLE {SILVER}.silver_order_reviews AS
WITH r AS (
    SELECT
        review_id,
        order_id,
        CAST(TRY_CAST(review_score AS DOUBLE) AS INT) AS review_score,
        NULLIF(TRIM(review_comment_title), '')   AS review_comment_title,
        NULLIF(TRIM(review_comment_message), '') AS review_comment_message,
        TRY_CAST(NULLIF(TRIM(review_creation_date), '')    AS TIMESTAMP) AS review_creation_ts,
        TRY_CAST(NULLIF(TRIM(review_answer_timestamp), '') AS TIMESTAMP) AS review_answer_ts,
        _loaded_at
    FROM {BRONZE}.order_reviews
)
SELECT
    r.review_id, r.order_id, r.review_score,
    r.review_comment_title, r.review_comment_message,
    r.review_creation_ts, r.review_answer_ts,
    (r.review_comment_title IS NOT NULL OR r.review_comment_message IS NOT NULL) AS has_comment,
    (COUNT(*) OVER (PARTITION BY r.review_id) > 1) AS is_duplicate_review_id,
    ROW_NUMBER() OVER (PARTITION BY r.order_id
                       ORDER BY r.review_answer_ts DESC, r.review_creation_ts DESC, r.review_id) AS review_rank,
    r._loaded_at AS _bronze_loaded_at,
    current_timestamp() AS _silver_loaded_at
FROM r
""")

In [0]:
t = f"{SILVER}.silver_order_reviews"
check("lignes", scalar(f"SELECT COUNT(*) FROM {t}"), 99224)
check("notes hors 1-5", scalar(f"SELECT COUNT(*) FROM {t} WHERE review_score IS NULL OR review_score NOT BETWEEN 1 AND 5"), 0)
check("lignes is_duplicate_review_id", scalar(f"SELECT COUNT(*) FROM {t} WHERE is_duplicate_review_id"), 814 + 789)
check("commandes avec plusieurs avis", scalar(f"SELECT COUNT(DISTINCT order_id) FROM {t} WHERE review_rank = 2"), 551)
print("Moyenne des notes (attendu ~4.09) :", scalar(f"SELECT ROUND(AVG(review_score), 2) FROM {t}"))

## Non testé encore (à regarder, puis documenter)
Avis daté **avant** la livraison, ou réponse **avant** la création de l'avis.

In [0]:
spark.sql(f"""
    SELECT SUM(CASE WHEN r.review_creation_ts < o.order_delivered_ts THEN 1 ELSE 0 END) AS avis_avant_livraison,
           SUM(CASE WHEN r.review_answer_ts < r.review_creation_ts THEN 1 ELSE 0 END)  AS reponse_avant_creation
    FROM {t} r JOIN {SILVER}.silver_orders o ON r.order_id = o.order_id
""").show()

## Pour ton journal (à rédiger avec tes mots)
1. Pourquoi `TRY_CAST` plutôt que `CAST` ?
2. Pourquoi on garde les lignes anormales avec un indicateur `flag_…` ?
3. Pourquoi Silver a le même nombre de lignes que Bronze ?